In [3]:
import pandas as pd

In [5]:
DATA_PATH = "../data/raw/incident_event_log.csv"

df = pd.read_csv(DATA_PATH)

In [6]:
print("--- Exact Duplicate Rows ---")
print(df.duplicated().sum())

--- Exact Duplicate Rows ---
0


In [7]:
print("\n --- Suspicious Placeholder Values ---")
placeholders = ["?", "NA", "N/A", "NULL", "null", "None", "-100"]

for col in df.columns:
  values = df[col].astype(str)

  found = {}

  for placeholder in placeholders:
    count = (values == placeholder).sum()

    if count > 0:
      found[placeholder] = count

  if found:
    print(f"{col}: {found}")


 --- Suspicious Placeholder Values ---
incident_state: {'-100': np.int64(5)}
caller_id: {'?': np.int64(29)}
opened_by: {'?': np.int64(4835)}
sys_created_by: {'?': np.int64(53076)}
sys_created_at: {'?': np.int64(53076)}
location: {'?': np.int64(76)}
category: {'?': np.int64(78)}
subcategory: {'?': np.int64(111)}
u_symptom: {'?': np.int64(32964)}
cmdb_ci: {'?': np.int64(141267)}
assignment_group: {'?': np.int64(14213)}
assigned_to: {'?': np.int64(27496)}
problem_id: {'?': np.int64(139417)}
rfc: {'?': np.int64(140721)}
vendor: {'?': np.int64(141468)}
caused_by: {'?': np.int64(141689)}
closed_code: {'?': np.int64(714)}
resolved_by: {'?': np.int64(226)}
resolved_at: {'?': np.int64(3141)}


In [11]:
print("\n--- Important Categorical Values ---")

categorical_columns = [
  "incident_state",
  "contact_type",
  "impact",
  "urgency",
  "priority",
  "made_sla",
]

for col in categorical_columns:
  print(f"\n{col}")
  print(df[col].value_counts(dropna=False))


--- Important Categorical Values ---

incident_state
incident_state
Active                38716
New                   36407
Resolved              25751
Closed                24985
Awaiting User Info    14642
Awaiting Vendor         707
Awaiting Problem        461
Awaiting Evidence        38
-100                      5
Name: count, dtype: int64

contact_type
contact_type
Phone             140462
Self service         995
Email                220
IVR                   18
Direct opening        17
Name: count, dtype: int64

impact
impact
2 - Medium    134335
3 - Low         3886
1 - High        3491
Name: count, dtype: int64

urgency
urgency
2 - Medium    134094
1 - High        4020
3 - Low         3598
Name: count, dtype: int64

priority
priority
3 - Moderate    132452
4 - Low           4030
2 - High          2972
1 - Critical      2258
Name: count, dtype: int64

made_sla
made_sla
True     132497
False      9215
Name: count, dtype: int64


In [9]:
for col in categorical_columns:
  print(f"\n{col}")
  print(df[col].value_counts(dropna=False))


incident_state
incident_state
Active                38716
New                   36407
Resolved              25751
Closed                24985
Awaiting User Info    14642
Awaiting Vendor         707
Awaiting Problem        461
Awaiting Evidence        38
-100                      5
Name: count, dtype: int64

contact_type
contact_type
Phone             140462
Self service         995
Email                220
IVR                   18
Direct opening        17
Name: count, dtype: int64

impact
impact
2 - Medium    134335
3 - Low         3886
1 - High        3491
Name: count, dtype: int64

urgency
urgency
2 - Medium    134094
1 - High        4020
3 - Low         3598
Name: count, dtype: int64

priority
priority
3 - Moderate    132452
4 - Low           4030
2 - High          2972
1 - Critical      2258
Name: count, dtype: int64

made_sla
made_sla
True     132497
False      9215
Name: count, dtype: int64


In [12]:
print("\n--- Datetime Validation ---")

datetime_columns = [
  "opened_at",
  "closed_at",
  "resolved_at",
  "sys_created_at",
  "sys_updated_at",
]

for col in datetime_columns:
  parsed = pd.to_datetime(
    df[col],
    dayfirst=True,
    errors="coerce"
  )

  invalid = parsed.isnull().sum()

  print(
    f"{col}: "
    f"invalid={invalid:,}, "
    f"min={parsed.min()}, "
    f"max={parsed.max()}"
  )


--- Datetime Validation ---
opened_at: invalid=0, min=2016-02-29 01:16:00, max=2017-02-16 14:17:00
closed_at: invalid=0, min=2016-02-29 17:47:00, max=2017-02-18 15:00:00
resolved_at: invalid=3,141, min=2016-02-29 09:04:00, max=2017-02-17 00:47:00
sys_created_at: invalid=53,076, min=2016-02-29 01:23:00, max=2017-01-27 16:59:00
sys_updated_at: invalid=0, min=2016-02-29 01:23:00, max=2017-02-18 15:00:00


In [13]:
print("\n--- Duplicate Event Keys ---")

duplicate_event_keys = df.duplicated(
  subset=[
    "number",
    "sys_created_at",
    "sys_mod_count"
  ]
).sum()

print(f"Duplicate Event Keys: {duplicate_event_keys}")


--- Duplicate Event Keys ---
Duplicate Event Keys: 5


In [16]:
placeholder_counts = {}

for col in df.columns:
    values = df[col].astype(str)

    count = (
        (values == "?") |
        (values == "-100")
    ).sum()

    if count > 0:
        placeholder_counts[col] = {
            "count": count,
            "percentage": round(count / len(df) * 100, 2)
        }

placeholder_counts

{'incident_state': {'count': np.int64(5), 'percentage': np.float64(0.0)},
 'caller_id': {'count': np.int64(29), 'percentage': np.float64(0.02)},
 'opened_by': {'count': np.int64(4835), 'percentage': np.float64(3.41)},
 'sys_created_by': {'count': np.int64(53076), 'percentage': np.float64(37.45)},
 'sys_created_at': {'count': np.int64(53076), 'percentage': np.float64(37.45)},
 'location': {'count': np.int64(76), 'percentage': np.float64(0.05)},
 'category': {'count': np.int64(78), 'percentage': np.float64(0.06)},
 'subcategory': {'count': np.int64(111), 'percentage': np.float64(0.08)},
 'u_symptom': {'count': np.int64(32964), 'percentage': np.float64(23.26)},
 'cmdb_ci': {'count': np.int64(141267), 'percentage': np.float64(99.69)},
 'assignment_group': {'count': np.int64(14213),
  'percentage': np.float64(10.03)},
 'assigned_to': {'count': np.int64(27496), 'percentage': np.float64(19.4)},
 'problem_id': {'count': np.int64(139417), 'percentage': np.float64(98.38)},
 'rfc': {'count': np.i

In [17]:
important_categoricals = [
    "incident_state",
    "contact_type",
    "impact",
    "urgency",
    "priority",
    "made_sla",
    "knowledge",
    "u_priority_confirmation",
    "notify",
    "closed_code"
]

for col in important_categoricals:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- incident_state ---
incident_state
Active                38716
New                   36407
Resolved              25751
Closed                24985
Awaiting User Info    14642
Awaiting Vendor         707
Awaiting Problem        461
Awaiting Evidence        38
-100                      5
Name: count, dtype: int64

--- contact_type ---
contact_type
Phone             140462
Self service         995
Email                220
IVR                   18
Direct opening        17
Name: count, dtype: int64

--- impact ---
impact
2 - Medium    134335
3 - Low         3886
1 - High        3491
Name: count, dtype: int64

--- urgency ---
urgency
2 - Medium    134094
1 - High        4020
3 - Low         3598
Name: count, dtype: int64

--- priority ---
priority
3 - Moderate    132452
4 - Low           4030
2 - High          2972
1 - Critical      2258
Name: count, dtype: int64

--- made_sla ---
made_sla
True     132497
False      9215
Name: count, dtype: int64

--- knowledge ---
knowledge
False    116

In [18]:
datetime_cols = [
    "opened_at",
    "sys_created_at",
    "sys_updated_at",
    "resolved_at",
    "closed_at"
]

for col in datetime_cols:
    temp = df[col].replace("?", pd.NA)

    parsed = pd.to_datetime(
        temp,
        dayfirst=True,
        errors="coerce"
    )

    print(f"\n--- {col} ---")
    print("Invalid:", parsed.isna().sum())
    print("Earliest:", parsed.min())
    print("Latest:", parsed.max())


--- opened_at ---
Invalid: 0
Earliest: 2016-02-29 01:16:00
Latest: 2017-02-16 14:17:00

--- sys_created_at ---
Invalid: 53076
Earliest: 2016-02-29 01:23:00
Latest: 2017-01-27 16:59:00

--- sys_updated_at ---
Invalid: 0
Earliest: 2016-02-29 01:23:00
Latest: 2017-02-18 15:00:00

--- resolved_at ---
Invalid: 3141
Earliest: 2016-02-29 09:04:00
Latest: 2017-02-17 00:47:00

--- closed_at ---
Invalid: 0
Earliest: 2016-02-29 17:47:00
Latest: 2017-02-18 15:00:00


In [19]:
cols = [
    "impact",
    "urgency",
    "priority",
    "made_sla",
    "knowledge",
    "u_priority_confirmation",
    "notify",
    "closed_code"
]

for col in cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())


--- impact ---
impact
2 - Medium    134335
3 - Low         3886
1 - High        3491

--- urgency ---
urgency
2 - Medium    134094
1 - High        4020
3 - Low         3598

--- priority ---
priority
3 - Moderate    132452
4 - Low           4030
2 - High          2972
1 - Critical      2258

--- made_sla ---
made_sla
True     132497
False      9215

--- knowledge ---
knowledge
False    116349
True      25363

--- u_priority_confirmation ---
u_priority_confirmation
False    100740
True      40972

--- notify ---
notify
Do Not Notify    141593
Send Email          119

--- closed_code ---
closed_code
code 6     86583
code 7     20733
code 9     13562
code 8      5646
code 5      4469
code 1      3265
code 10     1678
code 11     1493
code 4      1139
code 16     1091
?            714
code 3       608
code 2       349
code 15      183
code 17      115
code 13       59
code 12       13
code 14       12


In [20]:
print("Exact duplicates:", df.duplicated().sum())

print(
    "Duplicate event keys:",
    df.duplicated(
        subset=["number", "sys_updated_at", "sys_mod_count"]
    ).sum()
)

Exact duplicates: 0
Duplicate event keys: 0
